# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haritharamadass/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)



## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The FlyRank paper reports that content trending upward was about 37.6% longer and 20% younger than content trending downward. The authors also describe this as an observational comparison rather than causal evidence.

**Methodology question:**  
How exactly was the growing/declining label created, and were the variables being compared measured independently from the window used to define that label? I would also want to know whether client, topic, and content type were controlled for, because these factors could influence both page length/age and search performance.

### Finding 2 — The Freshness Multiplier

The paper reports that content older than 365 days but refreshed within 30 days had about a 3.2x higher health score and 57x more impressions than older untouched content. The paper also warns that some very old freshness buckets have small and unstable samples.

**Methodology question:**  
Were refreshed and non-refreshed pages comparable before the refresh? Strong pages may be more likely to be selected for updating, so the difference could partly reflect selection bias. A matched comparison or before/after design would provide stronger evidence before treating refresh timing as the cause of the improvement.

In [1]:
import pandas as pd

# Recalculate the headline numbers reported in the paper
audit_check = pd.DataFrame({
    "Finding": [
        "Growing content — word count",
        "Growing content — age",
        "Freshness multiplier — health",
        "Freshness multiplier — impressions"
    ],
    "Calculation": [
        (3180 / 2311 - 1) * 100,
        (1 - 184 / 230) * 100,
        34.5 / 10.7,
        4039 / 71
    ],
    "Reported interpretation": [
        "% longer",
        "% younger",
        "x health",
        "x impressions"
    ]
})

audit_check.round(2)

,Finding,Calculation,Reported interpretation
0,Growing content — word count,37.60,% longer
1,Growing content — age,20.00,% younger
2,Freshness multiplier — health,3.22,x health
3,Freshness multiplier — impressions,56.89,x impressions


## 2. My model under an honest split (before/after)

### Stronger validation check

In Week 5, I already used an 80/20 grouped train-test split by `client_hash_id`, so content from the same client did not appear on both sides of the split.

That single held-out split produced a **Precision@20 of 0.800** for Logistic Regression.

For this audit, I make the validation more robust by using **5-fold GroupKFold by client**. Each fold tests the model on clients that were not used to train that fold. I report the result for every fold, the average Precision@20, and the held-out base rate.

This does not turn the result into a causal claim. It tests whether the ranking performance is reasonably stable across different unseen-client groups.

In [2]:
# ============================================================
# ML-09 — Section 2
# Stronger client-grouped validation of the Week-5 model
# ============================================================

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# Same model setup used in Week 5
# ------------------------------------------------------------

RANDOM_STATE = 42

FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]

TARGET = "declined_next_month"
GROUP = "client_hash_id"


# ------------------------------------------------------------
# Connect to the FlyRank warehouse
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")


# ------------------------------------------------------------
# March = information available at prediction time
# April = future month used only to construct the label
# ------------------------------------------------------------

MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""


# ------------------------------------------------------------
# Rebuild the Week-5 feature frame
# ------------------------------------------------------------

feature_frame = con.sql(f"""
WITH march AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,

        SUM(
            gsc_avg_position * gsc_impressions
        ) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS avg_position_31d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days

    FROM {APRIL}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.impressions_31d,

    100.0 * m.clicks_31d
        / NULLIF(m.impressions_31d, 0) AS ctr_31d,

    m.avg_position_31d,
    m.active_gsc_days,

    (
        a.april_impressions
        < 0.80 * m.impressions_31d
    ) AS declined_next_month

FROM march AS m

INNER JOIN april AS a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE
    m.impressions_31d >= 100
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
    AND m.avg_position_31d IS NOT NULL
    AND m.avg_position_31d > 0
    AND a.april_impressions IS NOT NULL
""").df()


# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

model_data = (
    feature_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=FEATURES + [TARGET, GROUP])
    .reset_index(drop=True)
)

X = model_data[FEATURES]
y = model_data[TARGET].astype(int)
groups = model_data[GROUP]


def precision_at_k(y_true, scores, k=20):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_k = np.argsort(scores)[::-1][:k]

    return y_true[top_k].mean()


# ------------------------------------------------------------
# 5-fold validation grouped by client
# ------------------------------------------------------------

gkf = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    overlap = len(train_groups.intersection(test_groups))

    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ])

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    fold_results.append({
        "Fold": fold,
        "Test rows": len(test_idx),
        "Test clients": len(test_groups),
        "Client overlap": overlap,
        "Base rate": y_test.mean(),
        "Precision@20": precision_at_k(
            y_test,
            probabilities,
            k=20
        )
    })


fold_results = pd.DataFrame(fold_results)

print("5-fold GroupKFold results")
print("-" * 60)

display(
    fold_results.round(3)
)

before_p20 = 0.800
after_p20 = fold_results["Precision@20"].mean()

comparison = pd.DataFrame({
    "Validation": [
        "Week-5 single grouped holdout",
        "ML-09 5-fold GroupKFold"
    ],
    "Precision@20": [
        before_p20,
        after_p20
    ]
})

print("\nBefore / after comparison")
print("-" * 60)

display(comparison.round(3))

print(
    "Mean GroupKFold base rate:",
    round(fold_results["Base rate"].mean(), 3)
)

print(
    "Mean GroupKFold Precision@20:",
    round(after_p20, 3)
)

print(
    "Maximum client overlap:",
    fold_results["Client overlap"].max()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5-fold GroupKFold results
------------------------------------------------------------


,Fold,Test rows,Test clients,Client overlap,Base rate,Precision@20
0,1,21620,1,0,0.431,0.70
1,2,19800,10,0,0.697,0.90
2,3,19800,13,0,0.540,0.50
3,4,19874,7,0,0.404,0.55
4,5,19799,12,0,0.510,0.90



Before / after comparison
------------------------------------------------------------


,Validation,Precision@20
0,Week-5 single grouped holdout,0.80
1,ML-09 5-fold GroupKFold,0.71


Mean GroupKFold base rate: 0.516
Mean GroupKFold Precision@20: 0.71
Maximum client overlap: 0


### Interpretation

The original Week-5 grouped holdout produced a Precision@20 of **0.800**. Under the stronger 5-fold client-grouped validation, the mean Precision@20 is **0.710**.

The lower score does not mean the model failed. It shows that the single Week-5 holdout was somewhat optimistic compared with performance across several different groups of unseen clients.

Performance also varies across folds (0.50 to 0.90), which suggests that the model generalizes better to some client groups than others. There is **zero client overlap** in every fold, so this comparison avoids client-level leakage.

I therefore treat **0.710 mean Precision@20** as the more cautious estimate of ranking performance rather than claiming that the model will always achieve 0.800.

## 3. Leakage audit

### Leakage review

My prediction point is the end of March 2026. The outcome is whether April impressions fall below 80% of March impressions.

I reviewed the final Week-5 feature set:

- `impressions_31d` — **potential label-related risk** because March impressions is also used when defining the decline threshold.
- `ctr_31d` — available during March; no April information is used.
- `avg_position_31d` — available during March; no future information is used.
- `active_gsc_days` — available during March; no future information is used.
- `client_hash_id` — used only for grouping, never as a model feature.
- `content_hash_id` — identifier only, never used as a model feature.
- April performance — used only to construct the future label, never as a feature.

Because `impressions_31d` is the strongest leakage concern, I test the model both **with and without this feature** under the same client-grouped validation.

In [3]:
# ============================================================
# ML-09 — Section 3
# Leakage audit: challenge the suspicious feature
# ============================================================

SAFE_FEATURES = [
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]

FULL_FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]


def grouped_cv_precision(features):
    results = []

    X_audit = model_data[features]
    y_audit = model_data[TARGET].astype(int)
    groups_audit = model_data[GROUP]

    gkf = GroupKFold(n_splits=5)

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X_audit, y_audit, groups_audit),
        start=1
    ):

        X_train = X_audit.iloc[train_idx]
        X_test = X_audit.iloc[test_idx]

        y_train = y_audit.iloc[train_idx]
        y_test = y_audit.iloc[test_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            (
                "logistic_regression",
                LogisticRegression(
                    max_iter=1000,
                    random_state=RANDOM_STATE
                )
            )
        ])

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results.append(
            precision_at_k(
                y_test,
                probabilities,
                k=20
            )
        )

    return results


with_impressions = grouped_cv_precision(FULL_FEATURES)
without_impressions = grouped_cv_precision(SAFE_FEATURES)


leakage_comparison = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "With impressions_31d": with_impressions,
    "Without impressions_31d": without_impressions
})

display(leakage_comparison.round(3))


summary = pd.DataFrame({
    "Feature set": [
        "Full Week-5 feature set",
        "Without impressions_31d"
    ],
    "Mean Precision@20": [
        np.mean(with_impressions),
        np.mean(without_impressions)
    ]
})

print("\nLeakage challenge summary")
print("-" * 50)

display(summary.round(3))


print("Columns used as model features:")
print(FULL_FEATURES)

print("\nIdentifiers used as features:")
print(
    [
        c for c in ["client_hash_id", "content_hash_id"]
        if c in FULL_FEATURES
    ]
)

print("\nFuture April columns used as features:")
print(
    [
        c for c in FULL_FEATURES
        if "april" in c.lower()
    ]
)

,Fold,With impressions_31d,Without impressions_31d
0,1,0.70,0.80
1,2,0.90,0.85
2,3,0.50,0.60
3,4,0.55,0.80
4,5,0.90,0.90



Leakage challenge summary
--------------------------------------------------


,Feature set,Mean Precision@20
0,Full Week-5 feature set,0.71
1,Without impressions_31d,0.79


Columns used as model features:
['impressions_31d', 'ctr_31d', 'avg_position_31d', 'active_gsc_days']

Identifiers used as features:
[]

Future April columns used as features:
[]


### Leakage audit interpretation

The leakage challenge does not show evidence that `impressions_31d` was artificially inflating model performance.

Mean Precision@20 increased from **0.710 with the full Week-5 feature set to 0.790 without `impressions_31d`**.

This is reassuring because the model still ranks future declines well without the feature most closely related to the label definition. I therefore prefer the reduced feature set (`ctr_31d`, `avg_position_31d`, and `active_gsc_days`) as the more conservative version for this audit.

The audit also confirms that neither client/content identifiers nor April future-performance columns are used as model inputs.

This does not prove that the model is free from every possible bias. It shows that the main feature-level leakage concern tested here does not explain the observed ranking performance.

In [4]:
# ============================================================
# ML-09 — Real failure examples
# Out-of-fold predictions using the safer reduced feature set
# ============================================================

audit_features = [
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]

X_error = model_data[audit_features]
y_error = model_data[TARGET].astype(int)
groups_error = model_data[GROUP]

oof_probability = np.full(len(model_data), np.nan)

gkf = GroupKFold(n_splits=5)

for train_idx, test_idx in gkf.split(
    X_error,
    y_error,
    groups_error
):

    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ])

    model.fit(
        X_error.iloc[train_idx],
        y_error.iloc[train_idx]
    )

    oof_probability[test_idx] = model.predict_proba(
        X_error.iloc[test_idx]
    )[:, 1]


error_examples = model_data[audit_features + [TARGET]].copy()

error_examples["predicted_probability"] = oof_probability

error_examples["predicted_class"] = (
    error_examples["predicted_probability"] >= 0.5
).astype(int)


false_positives = (
    error_examples[
        (error_examples[TARGET].astype(int) == 0) &
        (error_examples["predicted_class"] == 1)
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(5)
)

false_negatives = (
    error_examples[
        (error_examples[TARGET].astype(int) == 1) &
        (error_examples["predicted_class"] == 0)
    ]
    .sort_values(
        "predicted_probability",
        ascending=True
    )
    .head(5)
)


print("High-confidence false positives")
display(false_positives.round(3))

print("\nHigh-confidence false negatives")
display(false_negatives.round(3))

High-confidence false positives


,ctr_31d,avg_position_31d,active_gsc_days,declined_next_month,predicted_probability,predicted_class
30561,0.0,0.192,31,False,0.668,1
78938,0.0,0.513,31,False,0.667,1
44603,0.0,0.644,31,False,0.667,1
78695,0.0,0.749,31,False,0.667,1
79205,0.0,0.913,31,False,0.667,1



High-confidence false negatives


,ctr_31d,avg_position_31d,active_gsc_days,declined_next_month,predicted_probability,predicted_class
1719,15.584,46.130,30,True,0.0,0
100665,9.441,7.518,31,True,0.0,0
100662,7.843,3.020,24,True,0.0,0
19431,7.500,33.694,28,True,0.0,0
8295,7.157,5.800,17,True,0.0,0


### Failure analysis

The out-of-fold predictions show concrete cases where the model is confidently wrong.

Several false positives have **0% March CTR and a full month of active GSC data**. The model assigns them relatively high decline probabilities (about 0.67), but they do not decline in the following month. This suggests that weak click capture can look risky without necessarily leading to a future impression decline.

The false negatives show the opposite pattern. Some pages have relatively strong March CTR values, so the model assigns them very low decline probabilities, but they still decline in April. This shows that healthy-looking click behavior does not guarantee that future visibility will remain stable.

These failures reinforce that the model should be used for **ranking and decision support**, not as a certain prediction of what will happen to an individual page.

## 4. Claim rewrite

### Original claim

> The learned model improves the held-out ranking compared with my Week-4 rule baseline and provides a more useful ordering of content for review on unseen clients.

### Safer claim after the validation audit

In this dataset, Logistic Regression showed **measured ranking skill for identifying content at risk of next-month impression decline**.

The original single client-grouped holdout produced **Precision@20 = 0.800**, while 5-fold client-grouped validation produced a mean **Precision@20 = 0.710**. After removing the feature most closely related to the label definition (`impressions_31d`), mean Precision@20 was **0.790**.

These results suggest a useful **directional ranking signal**, but performance varies across unseen client groups and individual predictions include both false positives and false negatives.

I therefore treat the model as **decision-support for prioritizing content review**, not as proof that a page will decline or that any feature causes the decline.

In [5]:
# ============================================================
# ML-09 — Section 4
# Final public-safe claim summary
# ============================================================

claim_summary = pd.DataFrame({
    "Evidence": [
        "Week-5 single grouped holdout",
        "5-fold client GroupKFold",
        "Reduced feature set after leakage challenge"
    ],
    "Precision@20": [
        0.800,
        0.710,
        0.790
    ],
    "Safe interpretation": [
        "Observed on one held-out client split",
        "Measured average across multiple unseen-client groups",
        "Directional ranking signal without impressions_31d"
    ]
})

display(claim_summary)

print("\nFinal claim:")
print(
    "The model provides a measured directional ranking signal "
    "for decision-support, but performance varies across unseen "
    "client groups and should not be interpreted as causal or certain."
)

,Evidence,Precision@20,Safe interpretation
0,Week-5 single grouped holdout,0.80,Observed on one held-out client split
1,5-fold client GroupKFold,0.71,Measured average across multiple unseen-client...
2,Reduced feature set after leakage challenge,0.79,Directional ranking signal without impressions...



Final claim:
The model provides a measured directional ranking signal for decision-support, but performance varies across unseen client groups and should not be interpreted as causal or certain.
